# spinangle — gated spherical nGPT-JEPA vs. official LeWM (Colab GPU)

Baseline = **official LeWM, unchanged**. The big cell does the repo's **official** install (`uv` + isolated **Python 3.10** venv + full `stable-worldmodel[train,env]`), then reproduces LeWM (Phase 1) and trains/evals the nGPT-JEPA variants on its own planner. Everything runs through the venv interpreter, so Colab's pre-pinned system packages can't break the resolution.

**Setup:** Runtime → GPU. Start with `BENCH='tworoom'` + `EPOCHS=5`. Every step is loud and `check=True` — the cell **stops at the real error** instead of drawing an empty plot. Env eval renders headless via `xvfb` + EGL.

## ▶️ The big cell (edit config, run)

In [ ]:
#@title 🌀 spinangle: gated spherical nGPT-JEPA vs official LeWM — REAL install + run
# Official install path: uv + an isolated Python 3.10 venv + full stable-worldmodel[train,env].
# Colab's SYSTEM python has pre-pinned packages that break the full resolution; an
# isolated 3.10 venv (exactly what the repo's README uses) avoids that. Everything
# below runs through the venv interpreter PY, not Colab's kernel python.
import os, subprocess, sys, glob

# ----------------------------- config -----------------------------
BENCH    = "tworoom"   # tworoom (3.4G, lightest) | pusht (13G) | reacher (24G) | cube (46G)
EPOCHS   = 5           # 5 = quick validation; 100 = matched-compute comparison
VARIANTS = ["official_lewm", "gated_spherical"]   # + lewm_nosigreg, simple_spherical, fullish_residual,
#            gated_spherical_projector_sigreg, gated_spherical_memory, gated_spherical_ssm, ngpt_lr ...
GET_DATA = True
BRANCH   = "claude/upbeat-babbage-kbmgsr"
GH_TOKEN = ""          # only needed if the repo is private
# ------------------------------------------------------------------

DATACFG = {"tworoom": "tworoom", "pusht": "pusht", "reacher": "dmc", "cube": "ogb"}[BENCH]
H = "/content/stable-wm"; VENV = "/content/lewmenv"; PY = f"{VENV}/bin/python"
os.environ["STABLEWM_HOME"] = H
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"

def run(cmd, check=True):
    print(f"\n\033[1;36m$ {cmd}\033[0m", flush=True)
    return subprocess.run(cmd, shell=True, check=check).returncode

try:
    from google.colab import userdata
    GH_TOKEN = GH_TOKEN or (userdata.get("GITHUB_TOKEN") or "")
except Exception:
    pass

run("nvidia-smi -L || echo '⚠️  NO GPU — Runtime > Change runtime type > GPU'", check=False)

# 0) clone (repo is public; token only if private) -----------------------------
if not os.path.isdir("/content/spinangle/.git"):
    auth = f"{GH_TOKEN}@" if GH_TOKEN else ""
    run(f"git clone -b {BRANCH} https://{auth}github.com/turtlenottortoise/spinangle.git /content/spinangle")
else:
    run("cd /content/spinangle && git pull", check=False)
os.chdir("/content/spinangle")

# 1) system libs for headless env rendering during eval (pygame + MuJoCo) ------
run("apt-get -qq update && apt-get -qq install -y xvfb zstd ffmpeg patchelf "
    "libegl1 libgl1-mesa-glx libosmesa6 libglfw3 libglew2.2 >/dev/null 2>&1", check=False)

# 2) REAL install: uv + Python 3.10 venv + full [train,env] --------------------
run("pip install -q uv")
run("uv python install 3.10")
run(f"uv venv --python 3.10 {VENV}")
run(f"uv pip install --python {PY} 'stable-worldmodel[train,env]'")
run(f"uv pip install --python {PY} matplotlib huggingface_hub")          # used by our scripts
# ensure the venv torch sees the GPU; reinstall a CUDA build only if needed
gpu_ok = subprocess.run(f"{PY} -c \"import torch,sys; sys.exit(0 if torch.cuda.is_available() else 1)\"",
                        shell=True).returncode == 0
if not gpu_ok:
    run(f"uv pip install --python {PY} torch torchvision --index-url https://download.pytorch.org/whl/cu124")
run(f"{PY} -c \"import torch,hydra,stable_worldmodel,stable_pretraining as s; "
    f"print('torch',torch.__version__,'cuda',torch.cuda.is_available(),'| stack OK')\"")

# 3) harness sanity (under the venv) -------------------------------------------
run(f"{PY} smoke_test.py && {PY} metrics.py")

# 4) data + checkpoint; PHASE 1 reproduce official LeWM (eval renders -> xvfb) --
run(f"{PY} scripts/download_assets.py --benchmark {BENCH} --ckpt" + (" --data" if GET_DATA else ""))
run(f"xvfb-run -a {PY} eval.py --config-name={BENCH}.yaml policy={BENCH}/lewm")

# 5) PHASES 2-3: train + eval variants (same eval.py + planner) ----------------
CKPT = f"{H}/checkpoints/{BENCH}"
for v in VARIANTS:
    run(f"{PY} train.py +experiment={v} data={DATACFG} "
        f"output_model_name={BENCH}/{v} trainer.max_epochs={EPOCHS} wandb.enabled=false")
    for old in sorted(glob.glob(f"{CKPT}/{v}/weights_epoch_*.pt"), key=os.path.getmtime)[:-1]:
        os.remove(old)               # keep newest ckpt so load_pretrained finds one .pt
    run(f"xvfb-run -a {PY} eval.py --config-name={BENCH}.yaml policy={BENCH}/{v}")
    sph = "" if v in ("official_lewm", "lewm_nosigreg") else "--spherical"
    run(f"{PY} scripts/eval_latent_metrics.py --policy {BENCH}/{v} --data {DATACFG} "
        f"--benchmark {BENCH} --variant {v} {sph} --horizon 20 --num_batches 16", check=False)

# 6) plots ---------------------------------------------------------------------
run(f"{PY} scripts/make_plots.py")
from IPython.display import Image, display
for p in ["success_vs_steps", "rollout_error_vs_horizon", "retrieval_vs_steps",
          "rank_clumping", "planning_budget_curve"]:
    fp = f"/content/spinangle/plots/{p}.png"
    if os.path.exists(fp):
        display(Image(fp))
print("\n✅ DONE — results in results/all_runs.csv, plots in plots/. Fill report.md.")


## Phase 7 — νGPT scaling (optional; run after the loop above)

In [ ]:
PY, BENCH, DATACFG, EPOCHS = '/content/lewmenv/bin/python', 'tworoom', 'tworoom', 100
for v in ['gated_spherical', 'ngpt_lr', 'ngpt_lr_groups']:
    !{PY} train.py +experiment={v} data={DATACFG} output_model_name={BENCH}/{v} \
        trainer.max_epochs={EPOCHS} wandb.enabled=false
    !xvfb-run -a {PY} eval.py --config-name={BENCH}.yaml policy={BENCH}/{v}


## Persist results back to the branch (optional)

In [ ]:
!cd /content/spinangle && git add results/all_runs.csv plots/*.png && \
  git -c user.email=colab@local -c user.name=colab commit -m 'colab: results' && \
  git push || echo 'configure git auth (token) to push'
